# Feature Engineering

After completing data cleaning and exploratory analysis, feature engineering becomes the crucial step that transforms raw attributes into meaningful inputs for modeling. By encoding categorical variables, scaling numeric features, and deriving clinically relevant labels, we ensure that the dataset is balanced, standardized, and optimized for machine learning. This process enhances model accuracy, prevents bias from uneven feature ranges, and captures the underlying patterns that drive predictive performance.


**1. Import libraries & setup**

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

**2. Load data**

In [0]:
# load data to dataframe and display 
df = spark.table('workspace.default.diabetes_data_explored')
display(df.limit(5))

cholesterol_total,stabilized_glucose,hdl_cholesterol,chol_hdl_ratio,location,age,gender,body_frame,systolic_bp_1,diastolic_bp_1,minutes_post_meal,BMI,waist_hip_ratio,hba1c_category
203.0,82.0,56.0,3.6,Buckingham,46,female,medium,118.0,59.0,720,22.130944261888523,0.7631578947368421,Normal
165.0,97.0,24.0,6.9,Buckingham,29,female,large,112.0,68.0,360,37.41920003371257,0.9583333333333334,Normal
228.0,92.0,37.0,6.2,Buckingham,58,female,large,190.0,92.0,180,48.37024068028249,0.8596491228070176,Normal
78.0,93.0,12.0,6.5,Buckingham,67,male,large,110.0,50.0,480,18.637828409539644,0.868421052631579,Normal
249.0,90.0,28.0,8.9,Buckingham,64,male,medium,138.0,80.0,300,27.824746566448155,1.0731707317073171,Diabetic


**3. Split data**

In [0]:
# Split the data
splits = df.randomSplit([0.7, 0.3])
train = splits[0]
test = splits[1]
print ("Training Rows:", train.count(), " Testing Rows:", test.count())

Training Rows: 262  Testing Rows: 125


**4. Encode Categorical Columns**

Encode the `hba1c_category`,  `location`, `gender` and `body_frame` categorical column values as numeric.

In [0]:
# Define indexers for each categorical column
indexer_label = StringIndexer(inputCol="hba1c_category", outputCol="label")
indexer_location = StringIndexer(inputCol="location", outputCol="locationIdx")
indexer_gender = StringIndexer(inputCol="gender", outputCol="genderIdx")
indexer_body_frame = StringIndexer(inputCol="body_frame", outputCol="bodyFrameIdx")

# Chain them together in a pipeline
pipeline = Pipeline(stages=[indexer_label, indexer_location, indexer_gender, indexer_body_frame])

# Fit and transform the training data
indexedData = (pipeline.fit(train).transform(train).drop("location", "gender", "body_frame", "minutes_post_meal"))

display(indexedData.limit(5))

cholesterol_total,stabilized_glucose,hdl_cholesterol,chol_hdl_ratio,age,systolic_bp_1,diastolic_bp_1,BMI,waist_hip_ratio,hba1c_category,label,locationIdx,genderIdx,bodyFrameIdx
78.0,93.0,12.0,6.5,67,110.0,50.0,18.637828409539644,0.868421052631579,Normal,0.0,1.0,1.0,1.0
118.0,95.0,39.0,3.0,47,140.0,76.0,21.112667908929566,0.8333333333333334,Normal,0.0,0.0,0.0,2.0
122.0,82.0,43.0,2.8,36,110.0,80.0,25.523036723518402,0.9111111111111111,Normal,0.0,0.0,0.0,0.0
129.0,110.0,42.0,3.1,56,140.0,75.0,19.387037970569732,0.8947368421052632,Prediabetic,2.0,1.0,1.0,2.0
132.0,83.0,40.0,3.3,28,136.0,86.0,34.210753975141174,0.7884615384615384,Normal,0.0,0.0,0.0,0.0


In [0]:
# show hba1c_category encoded mapping
display(indexedData.select("hba1c_category", "label").distinct())

hba1c_category,label
Prediabetic,2.0
Diabetic,1.0
Normal,0.0


In [0]:
#drop hba1c_category column
indexedData = indexedData.drop("hba1c_category")

**5. Normalize Numeric Features**

Numeric values often exist on different ranges. During model training, the absolute units of measurement matter less than the relative differences between observations. If one feature has much larger values, it can dominate the learning process and distort the model’s predictions. To prevent this imbalance, numeric features are typically scaled to a common range — for example, mapping values to decimals between 0.0 and 1.0.

In [0]:
# Create a vector column containing all numeric features
numericFeatures = [c for c in indexedData.columns if c not in ["label", "locationIdx", "genderIdx", "bodyFrameIdx"]]
numericColVector = VectorAssembler(inputCols=numericFeatures, outputCol="numericFeatures")
vectorizedData = numericColVector.transform(indexedData)


In [0]:
# Use a MinMax scaler to normalize the numeric values in the vector
minMax = MinMaxScaler(inputCol = numericColVector.getOutputCol(), outputCol="normalizedFeatures")
scaledData = minMax.fit(vectorizedData).transform(vectorizedData)

In [0]:
# Display the data with numeric feature vectors (before and after scaling) 
display(scaledData.select("numericFeatures", "normalizedFeatures").limit(5))

numericFeatures,normalizedFeatures
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""78.0"",""93.0"",""12.0"",""6.5"",""67.0"",""110.0"",""50.0"",""18.637828409539644"",""0.868421052631579""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""0.12933753943217666"",""0.0"",""0.6329114229666516"",""0.6575342465753424"",""0.0847457627118644"",""0.0"",""0.07448523496384489"",""0.4047442550037067""]}"
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""118.0"",""95.0"",""39.0"",""3.0"",""47.0"",""140.0"",""76.0"",""21.112667908929566"",""0.8333333333333334""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.1486988847583643"",""0.13564668769716087"",""0.25"",""0.18987342688999548"",""0.3835616438356164"",""0.3389830508474576"",""0.3611111111111111"",""0.14446093872280474"",""0.3286384976525824""]}"
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""122.0"",""82.0"",""43.0"",""2.799999952316284"",""36.0"",""110.0"",""80.0"",""25.523036723518402"",""0.9111111111111111""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.16356877323420074"",""0.0946372239747634"",""0.28703703703703703"",""0.16455696393541572"",""0.2328767123287671"",""0.0847457627118644"",""0.41666666666666663"",""0.26916343426382927"",""0.49733959311424114""]}"
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""129.0"",""110.0"",""42.0"",""3.0999999046325684"",""56.0"",""140.0"",""75.0"",""19.387037970569732"",""0.8947368421052632""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.1895910780669145"",""0.1829652996845426"",""0.2777777777777778"",""0.20253164327750114"",""0.5068493150684932"",""0.3389830508474576"",""0.3472222222222222"",""0.0956690193300414"",""0.46182357301704985""]}"
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""132.0"",""83.0"",""40.0"",""3.299999952316284"",""28.0"",""136.0"",""86.0"",""34.210753975141174"",""0.7884615384615384""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.2007434944237918"",""0.09779179810725552"",""0.25925925925925924"",""0.22784810623208088"",""0.1232876712328767"",""0.3050847457627119"",""0.5"",""0.514807294761054"",""0.23131094257854828""]}"


In [0]:
#Prepare feature and labels for training
featVect = VectorAssembler(inputCols=["locationIdx", "genderIdx", "bodyFrameIdx", "normalizedFeatures"], outputCol="featuresVector")
preppedData = featVect.transform(scaledData)[col("featuresVector").alias("features"), col("label").alias("label")]
display(preppedData.limit(5))


features,label
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.0"",""1.0"",""1.0"",""0.0"",""0.12933753943217666"",""0.0"",""0.6329114229666516"",""0.6575342465753424"",""0.0847457627118644"",""0.0"",""0.07448523496384489"",""0.4047442550037067""]}",0.0
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""0.0"",""2.0"",""0.1486988847583643"",""0.13564668769716087"",""0.25"",""0.18987342688999548"",""0.3835616438356164"",""0.3389830508474576"",""0.3611111111111111"",""0.14446093872280474"",""0.3286384976525824""]}",0.0
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""0.0"",""0.0"",""0.16356877323420074"",""0.0946372239747634"",""0.28703703703703703"",""0.16455696393541572"",""0.2328767123287671"",""0.0847457627118644"",""0.41666666666666663"",""0.26916343426382927"",""0.49733959311424114""]}",0.0
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.0"",""1.0"",""2.0"",""0.1895910780669145"",""0.1829652996845426"",""0.2777777777777778"",""0.20253164327750114"",""0.5068493150684932"",""0.3389830508474576"",""0.3472222222222222"",""0.0956690193300414"",""0.46182357301704985""]}",2.0
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""0.0"",""0.0"",""0.2007434944237918"",""0.09779179810725552"",""0.25925925925925924"",""0.22784810623208088"",""0.1232876712328767"",""0.3050847457627119"",""0.5"",""0.514807294761054"",""0.23131094257854828""]}",0.0



The dataset has been fully prepared for machine learning.  
- Categorical fields were indexed.  
- Numeric features were normalized.  
- A diabetes status label was derived from `hba1c_percent`.  
- All predictors were assembled into a single feature vector.  

We now have a clean structure of **features** and **labels**, ready to move into the machine learning phase for training and evaluation.


## Model Training

In [0]:
#Train the machine learning model
lr = LogisticRegression(labelCol="label", featuresCol="features", maxIter=10, regParam=0.3)
model = lr.fit(preppedData)
print ("Model trained!")


Model trained!


In [0]:

# Prepare the test data for testing the model
# Fit and transform the test data
indexedTestData = (pipeline.fit(test).transform(test).drop("location", "gender", "body_frame", "hba1c_category"))
vectorizedTestData = numericColVector.transform(indexedTestData)
scaledTestData = minMax.fit(vectorizedTestData).transform(vectorizedTestData)
preppedTestData = featVect.transform(scaledTestData)[col("featuresVector").alias("features"), col("label").alias("label")]


In [0]:
# Get predictions
prediction = model.transform(preppedTestData)
predicted = prediction.select("features", "probability", col("prediction").astype("Int"), col("label").alias("trueLabel"))
display(predicted.limit(5))

features,probability,prediction,trueLabel
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.0"",""1.0"",""0.0"",""0.0"",""0.5192878338278932"",""0.12987012987012989"",""0.31000002169609125"",""0.640625"",""0.125"",""0.2631578947368421"",""0.34950638217327523"",""0.8392448221464552""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.4435839055576092"",""0.41011396158533714"",""0.14630213285705365""]}",0,1.0
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.0"",""0.0"",""2.0"",""0.014492753623188406"",""0.1513353115727003"",""0.25974025974025977"",""0.17000000882148764"",""0.03125"",""0.1375"",""0.18421052631578946"",""0.28274446402296144"",""0.6129967940562817""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8827333002260704"",""0.05373817182977348"",""0.06352852794415619""]}",0,0.0
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""1.0"",""2.0"",""0.043478260869565216"",""1.0"",""0.2207792207792208"",""0.23000000071525578"",""0.484375"",""0.30000000000000004"",""0.23684210526315788"",""0.21213883387212326"",""0.5982814751561081""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.15332404012218936"",""0.7675454290443013"",""0.07913053083350932""]}",1,1.0
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""1.0"",""1.0"",""0.05434782608695652"",""0.9584569732937686"",""0.4155844155844156"",""0.08999998784065218"",""0.765625"",""0.30000000000000004"",""0.4473684210526315"",""0.19519051296938675"",""0.4621647753294995""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.17211579897962365"",""0.7584357393389562"",""0.06944846168141995""]}",1,0.0
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""1.0"",""1.0"",""0.057971014492753624"",""0.09792284866468844"",""0.18181818181818182"",""0.28999999260902387"",""0.171875"",""0.17500000000000002"",""0.39473684210526316"",""0.13378518666534808"",""0.31728665207877443""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9243384001459042"",""0.04138807560201154"",""0.034273524252084235""]}",0,0.0


## Model Evaluation

In [0]:
# Get evaluation metrics for the classification model
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

# Simple accuracy
accuracy = evaluator.evaluate(prediction, {evaluator.metricName:"accuracy"})
print("Accuracy:", accuracy)

# Individual class metrics
labels = [0,1,2]
print("\nIndividual class metrics:")
for label in sorted(labels):
   print ("Class %s" % (label))

   # Precision
   precision = evaluator.evaluate(prediction, {evaluator.metricLabel:label,
                                               evaluator.metricName:"precisionByLabel"})
   print("\tPrecision:", precision)

   # Recall
   recall = evaluator.evaluate(prediction, {evaluator.metricLabel:label,
                                            evaluator.metricName:"recallByLabel"})
   print("\tRecall:", recall)

   # F1 score
   f1 = evaluator.evaluate(prediction, {evaluator.metricLabel:label,
                                        evaluator.metricName:"fMeasureByLabel"})
   print("\tF1 Score:", f1)

# Weighted (overall) metrics
overallPrecision = evaluator.evaluate(prediction, {evaluator.metricName:"weightedPrecision"})
print("Overall Precision:", overallPrecision)
overallRecall = evaluator.evaluate(prediction, {evaluator.metricName:"weightedRecall"})
print("Overall Recall:", overallRecall)
overallF1 = evaluator.evaluate(prediction, {evaluator.metricName:"weightedFMeasure"})
print("Overall F1 Score:", overallF1)




Accuracy: 0.856

Individual class metrics:
Class 0
	Precision: 0.8648648648648649
	Recall: 0.9795918367346939
	F1 Score: 0.9186602870813397
Class 1
	Precision: 0.7857142857142857
	Recall: 0.55
	F1 Score: 0.6470588235294117
Class 2
	Precision: 0.0
	Recall: 0.0
	F1 Score: 0.0
Overall Precision: 0.8037683397683397
Overall Recall: 0.856
Overall F1 Score: 0.8237590768364762


**Model Evaluation Summary**

The model achieves an overall accuracy of ~85.6%, but performance varies significantly across classes.

For ***Class 0 (Normal)**, results are excellent: high precision (0.86), recall (0.98), and F1 (0.92) show the model is very reliable at identifying non‑diabetic patients.

For **Class 1 (Diabetic)**, performance is moderate: precision (0.79) is decent, but recall (0.55) indicates many diabetic cases are missed, leading to a weaker F1 (0.65).

For **Class 2 (Prediabetic)**, the model fails completely, with precision, recall, and F1 all at 0.0 — meaning it does not detect prediabetic patients at all.

Overall precision (0.80), recall (0.86), and F1 (0.82) look solid, but they are heavily skewed by the dominance of Class 0. The imbalance in the dataset causes the model to favor the majority class while neglecting minority classes, especially prediabetics.